# Late Delivery Classification Baseline

A small baseline that predicts at order-creation time whether a shipment will be delivered late. It is trained on the gold star schema: shipment attributes from `logistics.gold.fact_shipment` joined to carrier attributes in `logistics.gold.dim_carrier` on `carrier_sk`, uses a time-based (leakage-safe) train/test split, and logs parameters, metrics and the model to MLflow.

**Label (proxy):** there are no usable ON_TIME/DELAYED labels: delivery timestamps fall before `created_ts`, and `promised_delivery_ts` is always `created_ts` + 5 min. So `is_late = 1` means the delivery attempt for the shipment was FAILED or RESCHEDULED (from `logistics.gold.fact_delivery_attempt`, joined on `shipment_sk`). That is a direct cause of late delivery. Swap in a true late flag once valid delivery timestamps exist.

In [0]:
from pyspark.sql import functions as F

# Gold star schema: fact_shipment -> dim_carrier on carrier_sk (sha2 surrogate key)
fact = spark.table("logistics.gold.fact_shipment")
carrier = spark.table("logistics.gold.dim_carrier")

# Proxy label: one delivery attempt per shipment; FAILED / RESCHEDULED => late
labels = (
    spark.table("logistics.gold.fact_delivery_attempt")
    .select("shipment_sk", (F.col("attempt_result") != "DELIVERED").cast("int").alias("is_late"))
    .dropDuplicates(["shipment_sk"])
)

# Only features known at order creation (no tracking, status, attempt-count or delivery fields => no leakage)
model_df = (
    fact.join(labels, "shipment_sk", "inner")
    .join(carrier.select("carrier_sk", "carrier_tier", "contract_type",
                         F.col("service_region").alias("carrier_region")), "carrier_sk", "left")
    .select(
        "shipment_id", "created_ts",
        "service_level", "destination_city", "carrier_tier", "contract_type", "carrier_region",
        "package_weight_kg", "shipping_cost",
        F.hour("created_ts").alias("created_hour"),
        F.dayofweek("created_ts").alias("created_dow"),
        "is_late",
    )
)

pdf = model_df.toPandas().sort_values("created_ts").reset_index(drop=True)
print("Rows:", pdf.shape, "| created_ts range:", pdf.created_ts.min(), "->", pdf.created_ts.max())
display(pdf.head(5))

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

CAT_COLS = ["service_level", "destination_city", "carrier_tier", "contract_type", "carrier_region"]
NUM_COLS = ["package_weight_kg", "shipping_cost", "created_hour", "created_dow"]
TARGET = "is_late"

print(pdf[TARGET].value_counts(normalize=True).round(3))
pdf[NUM_COLS].info()
display(pdf[NUM_COLS].describe())

display(pd.DataFrame({
    "feature": CAT_COLS + NUM_COLS,
    "type": ["categorical/nominal"] * 5 + ["numeric float", "numeric float", "numeric integer", "numeric integer"],
}))

# Leakage screen: single-feature AUC for numeric features
print({c: round(roc_auc_score(pdf[TARGET], pdf[c].fillna(pdf[c].mean())), 3) for c in NUM_COLS})

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
pdf[TARGET].value_counts().sort_index().plot.bar(ax=ax[0], title="Class frequency (1 = late)")
pdf.boxplot(column="package_weight_kg", by=TARGET, ax=ax[1])
pdf.groupby("service_level")[TARGET].mean().plot.bar(ax=ax[2], title="Late rate by service level")
ax[3].imshow(pdf[NUM_COLS + [TARGET]].corr(), cmap="coolwarm", vmin=-1, vmax=1)
ax[3].set_xticks(range(5)); ax[3].set_xticklabels(NUM_COLS + [TARGET], rotation=45)
ax[3].set_yticks(range(5)); ax[3].set_yticklabels(NUM_COLS + [TARGET]); ax[3].set_title("Correlogram")
plt.suptitle(""); plt.tight_layout(); plt.show()

In [0]:
# created_ts is the real order-creation time: hold out the latest 20% of orders to mirror
# scoring future shipments. Split on a timestamp cutoff so tied timestamps stay together.
cutoff = pdf["created_ts"].quantile(0.8)
train = pdf[pdf.created_ts < cutoff]
test = pdf[pdf.created_ts >= cutoff]

X_train, y_train = train[CAT_COLS + NUM_COLS], train[TARGET]
X_test, y_test = test[CAT_COLS + NUM_COLS], test[TARGET]

print(f"Cutoff: {cutoff} | train={len(train)} (late rate {y_train.mean():.2f}) | test={len(test)} (late rate {y_test.mean():.2f})")
assert train.created_ts.max() < test.created_ts.min()
assert set(train.shipment_id).isdisjoint(test.shipment_id)

In [0]:
import mlflow
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, confusion_matrix,
                             ConfusionMatrixDisplay)

SEED = 42
params = {"model": "LogisticRegression", "C": 1.0, "max_iter": 1000, "random_state": SEED,
          "split": "temporal_80_20_on_created_ts", "label": "attempt_result in (FAILED, RESCHEDULED)",
          "cat_features": ",".join(CAT_COLS), "num_features": ",".join(NUM_COLS)}

def build_pipeline():
    pre = ColumnTransformer([
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
        ("num", Pipeline([("imp", SimpleImputer(strategy="mean")),
                          ("sc", StandardScaler())]), NUM_COLS),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(C=params["C"], max_iter=params["max_iter"],
                                                               random_state=SEED))])

mlflow.sklearn.autolog(disable=True)
with mlflow.start_run(run_name="late_delivery_logreg_baseline") as run:
    mlflow.set_tags({"project": "logistics", "layer": "gold", "task": "late_delivery_classification"})
    mlflow.log_params(params)

    # Holdout evaluation (pipeline fit on train only)
    model = build_pipeline().fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    metrics = {
        "test_accuracy": accuracy_score(y_test, pred),
        "test_precision": precision_score(y_test, pred, zero_division=0),
        "test_recall": recall_score(y_test, pred, zero_division=0),
        "test_f1": f1_score(y_test, pred, zero_division=0),
        "test_roc_auc": roc_auc_score(y_test, proba),
        "test_pr_auc": average_precision_score(y_test, proba),
        "test_log_loss": log_loss(y_test, proba),
        "majority_class_baseline_accuracy": max(y_test.mean(), 1 - y_test.mean()),
        "n_train": len(X_train), "n_test": len(X_test),
    }

    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred)).plot(ax=ax)
    ax.set_title("Holdout confusion matrix")
    mlflow.log_figure(fig, "confusion_matrix.png")

    # Refit on all rows in the window and log as the final model artifact
    X_all, y_all = pdf[CAT_COLS + NUM_COLS], pdf[TARGET]
    final_model = build_pipeline().fit(X_all, y_all)
    metrics["n_final_fit"] = len(X_all)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_all.head(100), final_model.predict(X_all.head(100)))
    model_info = mlflow.sklearn.log_model(final_model, name="model", signature=signature,
                                          input_example=X_all.head(3))

print("Run ID:", run.info.run_id, "| Model URI:", model_info.model_uri)
display(pd.DataFrame(list(metrics.items()), columns=["metric", "value"]).round(3))